# Post hoc analysis
This notebook is the main entry point for reusable post hoc analyses.
Import analysis functions from `code/src/posthoc_analysis` and run them from here.

In [ ]:
from pathlib import Path
import sys

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.config as config
importlib.reload(config)

from posthoc_analysis.config import get_subject_group, BCI_GROUP_SUBJECTS, CONTROL_GROUP_SUBJECTS
print('BCI subjects:', BCI_GROUP_SUBJECTS)
print('Control subjects:', CONTROL_GROUP_SUBJECTS)

# Training trigger RT computation
Load one random training `triggers.txt` file, validate its structure, and compute reaction time for each trial.

In [ ]:
import random
from pathlib import Path
import sys

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.triggers as triggers
importlib.reload(triggers)

from posthoc_analysis.config import PROJECT_ROOT
from posthoc_analysis.triggers import (
    load_training_trigger_file,
    compute_training_reaction_times,
)

training_trigger_files = sorted(
    PROJECT_ROOT.rglob('*_training.triggers.txt')
)
if not training_trigger_files:
    raise FileNotFoundError(
        f"No training trigger files found under PROJECT_ROOT={PROJECT_ROOT}."
    )

chosen_file = random.choice(training_trigger_files)
print('Selected file:', chosen_file)

triggers_df = load_training_trigger_file(chosen_file)
rt_df = compute_training_reaction_times(triggers_df)

print(f"Loaded {len(triggers_df)} trigger rows")
print(f"Computed RT for {len(rt_df)} trials")
rt_df.head(10)


In [ ]:
from posthoc_analysis.triggers import rt_outlier_summary

outlier_stats = rt_outlier_summary(rt_df)
print(f"Reaction time mean: {outlier_stats['mean_rt_ms']:.1f} ms")
print(f"Reaction time std: {outlier_stats['std_rt_ms']:.1f} ms")
print(f"Percent beyond 2 SD: {outlier_stats['percent_outlier']:.1f}%")
print(f"Num beyond 2 SD: {outlier_stats['outlier_count']} of {outlier_stats['total_trials']}")
outlier_stats

# Training analysis file loading
Load one random training `analysis.txt` file and validate its structure.

In [ ]:
import random
from pathlib import Path
import sys

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.analysis as analysis
importlib.reload(analysis)

from posthoc_analysis.config import PROJECT_ROOT
from posthoc_analysis.analysis import load_training_analysis_file

training_analysis_files = sorted(
    PROJECT_ROOT.rglob('*_training.analysis.txt')
)
if not training_analysis_files:
    raise FileNotFoundError(
        f"No training analysis files found under PROJECT_ROOT={PROJECT_ROOT}."
    )

chosen_analysis_file = random.choice(training_analysis_files)
print('Selected analysis file:', chosen_analysis_file)

analysis_df = load_training_analysis_file(chosen_analysis_file)

print(f"Loaded {len(analysis_df)} trials")
analysis_df.head(10)

# Consolidated Trial-wise Data
Generate a single CSV file containing trial-wise data for all subjects, sessions, and training runs (excluding training_practice).
Each row represents one trial with metadata and merged information from trigger and analysis files.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.consolidated as consolidated
importlib.reload(consolidated)

from posthoc_analysis.consolidated import generate_consolidated_training_csv

# Generate the consolidated CSV
result = generate_consolidated_training_csv()

# Show a summary
print(f"CSV saved to: {result['output_path']}")
print()
print(f"Final DataFrame shape: {result['dataframe'].shape}")
if len(result['dataframe']) > 0:
    print(f"Column names: {list(result['dataframe'].columns)}")
    print()
    print("First 5 rows:")
    result['dataframe'].head()


# Comprehensive File Validation
Validate all trigger and analysis files against documentation specifications (file_contents.md, triggers.md, experiment_structure.md).

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.consolidated as consolidated
importlib.reload(consolidated)

from posthoc_analysis.consolidated import validate_all_files_comprehensive

# Run comprehensive validation
validate_all_files_comprehensive()

# Behavioral Summary Table
Create subject-level behavioral summary table for accuracy and timeout analyses.
One row per subject × group × session × trial_type with aggregated metrics.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)

from posthoc_analysis.behavioral import (
    load_and_summarize_behavioral_data,
    print_behavioral_summary_checks
)

# Load consolidated data and create behavioral summary table
behavioral_summary = load_and_summarize_behavioral_data()

print("Behavioral summary table created successfully!")
print(f"Summary table shape: {behavioral_summary['summary_table'].shape}")
print(f"Expected: 16 subjects × 2 sessions × 3 trial_types = 96 rows")
print()

# Show table structure
print("Summary table columns:")
print(list(behavioral_summary['summary_table'].columns))
print()

# Show sample data (first few rows)
print("Sample of behavioral summary table (first 12 rows):")
print(behavioral_summary['summary_table'].head(12))
print()

# Run validation
validation = behavioral_summary['validation']
print("Validation Results:")
print(f"Valid: {validation['valid']}")
if validation['issues']:
    print("Issues found:")
    for issue in validation['issues']:
        print(f"  - {issue}")
else:
    print("No validation issues found.")
print()

# Print key summary stats
summary_stats = validation['summary']
print("Key Summary Statistics:")
print(f"  Total rows: {summary_stats['total_rows']}")
print(f"  Subjects: {summary_stats['subjects']}")
print(f"  Groups: {summary_stats['groups']}")
print(f"  Sessions: {summary_stats['sessions']}")
print(f"  Trial types: {summary_stats['trial_types']}")
print()

# Run detailed sanity checks
print("Running detailed sanity checks...")
print_behavioral_summary_checks(
    behavioral_summary['summary_table'],
    behavioral_summary['original_data']
)

# Analysis 2: Accuracy & Timeout
Analyze accuracy and timeout rates by trial type and group.
Includes 3-way ANOVA for accuracy (Time × Trial Type × Group),
2-way ANOVA for overall accuracy (Time × Group),
and 2-way ANOVA for timeout rate (Time × Group).

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)

from posthoc_analysis.behavioral import analyze_accuracy_and_timeout

# Run Analysis 2
analysis2_results = analyze_accuracy_and_timeout(behavioral_summary['summary_table'])

In [ ]:
# Display figures from analysis
import matplotlib.pyplot as plt

print("\n" + "="*80)
print("ACCURACY & TIMEOUT FIGURES")
print("="*80 + "\n")

# Figure 1: Accuracy by Trial Type
print("Figure 1: Accuracy by Trial Type (Distractor vs No Distractor)")
analysis2_results['figures']['accuracy_by_trial_type']

In [ ]:
# Figure 2: Overall Accuracy
print("\nFigure 2: Overall Accuracy")
analysis2_results['figures']['overall_accuracy']

In [ ]:
# Figure 3: Timeout Rate
print("\nFigure 3: Timeout Rate")
analysis2_results['figures']['timeout_rate']

# Analysis 1: Reaction Time
Analyze mean reaction time for correct responses during training after excluding RTs below 150 ms and removing outliers using mean \u00b1 3 SD within each subject and session across all correct trials combined. The cleaned trials are then summarized by distractor versus no-distractor condition. This section reports warnings for partial coverage, summarizes outlier removal, runs the planned Time \u00d7 Trial Type \u00d7 Group analysis, and generates publication-style RT figures.


In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)

from posthoc_analysis.behavioral import analyze_reaction_time


In [ ]:
# Run Analysis 1 on the consolidated trial-level training data
analysis1_results = analyze_reaction_time(behavioral_summary['original_data'])


In [ ]:
print("\n" + "="*80)
print("REACTION TIME SUMMARY TABLES")
print("="*80 + "\n")

print("Warnings:")
for warning in analysis1_results['warnings']:
    print(f"- {warning}")

print("\nAverage percent of trials removed per subject (mean \u00b1 SEM), including pooled pre+post:")
analysis1_results['exclusion_summary_by_session']

print("\nCell-level mean RT summary:")
analysis1_results['cell_summary']


In [ ]:
print("\nTrial-removal percentages by subject \u00d7 session:")
analysis1_results['subject_session_exclusion_report']


In [ ]:
print("\nFigure 1: Reaction Time for Distractor Trials")
analysis1_results['figures']['rt_distractor']


In [ ]:
print("\nFigure 2: Reaction Time for No-Distractor Trials")
analysis1_results['figures']['rt_no_distractor']


In [ ]:
print("\nFigure 3: Reaction Time Across All Trials")
analysis1_results['figures']['rt_combined_overall']


In [ ]:
print("\nFigure 4: RT Distribution Before Outlier Removal")
analysis1_results['figures']['rt_histogram_before_outlier_removal']


## Analysis 3: Distractor cost

Compute distractor cost as mean RT on distractor trials minus mean RT on no-distractor trials after applying the same RT inclusion and outlier-removal rules used for Analysis 1.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)

from posthoc_analysis.behavioral import analyze_distractor_cost


In [ ]:
# Run Analysis 3 on the consolidated trial-level training data
analysis3_results = analyze_distractor_cost(behavioral_summary['original_data'])


In [ ]:
print("\n" + "="*80)
print("DISTRACTOR COST SUMMARY TABLES")
print("="*80 + "\n")

print("Warnings:")
for warning in analysis3_results['warnings']:
    print(f"- {warning}")

print("\nAverage percent of trials removed per subject (mean ± SEM), including pooled pre+post:")
analysis3_results['exclusion_summary_by_session']

print("\nSubject-level distractor-cost table:")
analysis3_results['subject_session_cost']

print("\nCell-level distractor-cost summary:")
analysis3_results['cell_summary']

print("\nPlanned distractor-cost post hoc tests:")
analysis3_results['posthoc_tests']


In [ ]:
print("\nFigure 5: Distractor Cost")
analysis3_results['figures']['distractor_cost']


## Stroop: Consolidate behavioral files

Create one trial-level CSV across all non-practice Stroop behavioral files, validate documented run and trial counts, and save the consolidated table for downstream Stroop analyses.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.consolidated as consolidated
importlib.reload(consolidated)

from posthoc_analysis.consolidated import generate_consolidated_stroop_csv


In [ ]:
stroop_consolidation_results = generate_consolidated_stroop_csv()


In [ ]:
print("\n" + "=" * 80)
print("STROOP CONSOLIDATION SUMMARY")
print("=" * 80)
print(f"Output path: {stroop_consolidation_results['output_path']}")
print(f"Subjects present: {len(stroop_consolidation_results['subjects_present'])}")
print(f"Runs loaded: {stroop_consolidation_results['total_runs']}")
print(f"Trials loaded: {stroop_consolidation_results['total_trials']}")
print("Collection issues:")
for key, value in stroop_consolidation_results['collection_issues'].items():
    print(f"- {key}: {len(value)}")
print("Merge issues:")
for key, value in stroop_consolidation_results['merge_issues'].items():
    print(f"- {key}: {len(value)}")


## Stroop Analysis 1: Timeout summary

Summarize Stroop timeout exclusions by run and by subject-session, then report the overall average percent of timeout trials removed across all available subject-sessions.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)

from posthoc_analysis.behavioral import load_and_analyze_stroop_timeout_data


In [ ]:
stroop_analysis1_results = load_and_analyze_stroop_timeout_data()


In [ ]:
print("\n" + "=" * 80)
print("STROOP ANALYSIS 1 SUMMARY")
print("=" * 80)
print("Warnings:")
for warning in stroop_analysis1_results['warnings']:
    print(f"- {warning}")

print("\nSubject-session timeout summary (averaged across runs):")
stroop_analysis1_results['subject_session_summary']

print("\nOverall timeout summary:")
stroop_analysis1_results['overall_summary']


## Stroop Analysis 2: Accuracy

Compute Stroop accuracy after removing timeout trials, summarize congruent, incongruent, and overall accuracy at the subject-session level, generate training-style figures, and run the planned pre/post by group accuracy analyses when the local stats stack is available.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)

from posthoc_analysis.behavioral import load_and_analyze_stroop_accuracy_data


In [ ]:
stroop_analysis2_results = load_and_analyze_stroop_accuracy_data()


In [ ]:
print("\n" + "=" * 80)
print("STROOP ANALYSIS 2 SUMMARY")
print("=" * 80)
print("Warnings:")
for warning in stroop_analysis2_results['warnings']:
    print(f"- {warning}")

print("\nValidation:")
stroop_analysis2_results['validation']

print("\nSummary table:")
stroop_analysis2_results['summary_table']

print("\nStatistics:")
stroop_analysis2_results['stats']


## Stroop Analyses 3-4: Reaction time and Stroop effect

Run the Stroop RT and Stroop-effect analyses together using the same cleaned correct-trial dataset. This section reports incorrect-trial and exclusion summaries once, then shows the RT summaries and Stroop-effect summaries generated from the shared preprocessing workflow.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)

from posthoc_analysis.behavioral import (
    analyze_stroop_effect,
    load_and_analyze_stroop_reaction_time_data,
)


In [ ]:
stroop_analysis3_results = load_and_analyze_stroop_reaction_time_data()
stroop_analysis4_results = analyze_stroop_effect(rt_results=stroop_analysis3_results)


In [ ]:
print("\n" + "=" * 80)
print("STROOP ANALYSES 3-4 SUMMARY")
print("=" * 80)
print("Shared preprocessing warnings:")
for warning in stroop_analysis3_results['warnings']:
    print(f"- {warning}")

print("\nIncorrect-trial summary:")
stroop_analysis3_results['incorrect_trial_summary']

print("\nSubject exclusion summary:")
stroop_analysis3_results['subject_exclusion_summary']

print("\nOverall exclusion summary:")
stroop_analysis3_results['overall_exclusion_summary']

print("\nStroop RT subject-session summary:")
stroop_analysis3_results['subject_session_summary']

print("\nSubject-session Stroop effect summary:")
stroop_analysis4_results['subject_session_stroop_effect']

print("\nCell summary:")
stroop_analysis4_results['cell_summary']

print("\nStroop RT statistics:")
stroop_analysis3_results['stats']

print("\nStroop effect statistics:")
stroop_analysis4_results['stats_anova']
stroop_analysis4_results['posthoc_tests']
